In [ ]:
# Celda 1 - comprobar GPU e instalar lo que Colab no trae
!nvidia-smi
!pip install -q pydicom
import torch
assert torch.cuda.is_available(), "Sin GPU: en Colab, Entorno de ejecucion > Cambiar tipo > GPU"
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Celda 2 - traer el codigo del paquete
# Opcion A (mientras el repo sea privado): comprimir la carpeta copiloto-triaje
# en el equipo y subirla aqui. Pesa unos 100 KB.
import sys, pathlib, zipfile
from google.colab import files

subidos = files.upload()            # seleccionar copiloto-triaje.zip
nombre = next(iter(subidos))
with zipfile.ZipFile(nombre) as z:
    z.extractall('/content')
RAIZ = pathlib.Path('/content/copiloto-triaje')

# Opcion B (cuando el repo sea publico, hito H8): comentar lo de arriba y usar
# !git clone https://github.com/<usuario>/copiloto-triaje.git /content/copiloto-triaje

sys.path.insert(0, str(RAIZ / 'src'))
import copiloto_triaje
print('paquete cargado desde', RAIZ)

In [ ]:
# Celda 3 - Drive para lo que debe sobrevivir, y descarga del dataset desde Kaggle
import os, pathlib
from google.colab import drive, userdata

drive.mount('/content/drive')
PERSISTENTE = pathlib.Path('/content/drive/MyDrive/copiloto-triaje')
PERSISTENTE.mkdir(parents=True, exist_ok=True)

# Secretos de Colab (icono de la llave): KAGGLE_USERNAME y KAGGLE_KEY
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/datos
ZIP = '/content/datos/rsna-pneumonia-detection-challenge.zip'
print('descargado:', pathlib.Path(ZIP).stat().st_size / 1e9, 'GB')

In [ ]:
# Celda 4 - preparar los datos: zip -> PNG -> manifiestos (dentro de Colab)
!python {RAIZ}/scripts/preparar_datos.py --zip {ZIP} --raiz /content/trabajo

In [ ]:
# Celda 5 - entrenamiento completo con GPU, checkpoints en Drive
import torch, pandas as pd, pathlib
from torch.utils.data import DataLoader
from copiloto_triaje.config import Config
from copiloto_triaje.datos.dataset import DatasetRadiografias
from copiloto_triaje.modelo.arquitectura import construir_modelo
from copiloto_triaje.modelo.entrenamiento import entrenar

cfg = Config(raiz_datos=pathlib.Path('/content/trabajo'),
             zip_rsna=pathlib.Path(ZIP), lote=64)
dispositivo = torch.device('cuda')
DIR_CHECKPOINTS = PERSISTENTE / 'checkpoints'   # en Drive: sobrevive a la caida

def cargador(nombre, entrenamiento):
    df = pd.read_csv(cfg.dir_manifiestos / f'{nombre}.csv')
    ds = DatasetRadiografias(df, cfg.dir_png, cfg.tam_imagen, entrenamiento)
    return DataLoader(ds, batch_size=cfg.lote, shuffle=entrenamiento, num_workers=2)

modelo = construir_modelo(congelar_backbone=True, preentrenado=True)
resultado = entrenar(modelo, cargador('train', True), cargador('val', False),
                     cfg, dispositivo, DIR_CHECKPOINTS)
print('mejor recall en validacion:', resultado['mejor_recall'],
      '| epoca:', resultado['mejor_epoca'])

In [ ]:
# Celda 6 - umbral en VALIDACION, evaluacion UNICA en prueba, reporte a Drive
import json
from copiloto_triaje.modelo.evaluacion import elegir_umbral, metricas_binarias
from copiloto_triaje.modelo.entrenamiento import evaluar_epoca

y_val, s_val = evaluar_epoca(modelo, cargador('val', False), dispositivo)
umbral, m_val, paso = elegir_umbral(y_val, s_val, cfg.especificidad_objetivo,
                                    cfg.especificidad_minima, cfg.recall_objetivo)
print('umbral elegido en validacion:', umbral, '| regla aplicada:', paso)

# El conjunto de prueba se toca UNA sola vez, con el umbral ya congelado.
y_test, s_test = evaluar_epoca(modelo, cargador('test', False), dispositivo)
m_test = metricas_binarias(y_test, (s_test >= umbral).astype(int))
print(json.dumps(m_test, indent=2, ensure_ascii=False))

(PERSISTENTE / 'reporte_metricas.json').write_text(
    json.dumps({'umbral': umbral, 'regla': paso, 'validacion': m_val,
                'prueba': m_test, 'historial': resultado['historial']},
               indent=2, ensure_ascii=False), encoding='utf-8')
print('reporte y checkpoints guardados en', PERSISTENTE)